# Lesson 03 — The evaluation harness ⭐

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/03-evaluation-harness/colab.ipynb)

*The pivot point of the whole course: measure first, optimise second.*

**What you will learn**

- Why retrieval and generation must be measured *separately*
- How to build a golden question set without fooling yourself
- recall@k, precision@k, MRR and nDCG@k — derived and implemented by hand, then checked against `ragkit`
- The traps: ceiling effects, arithmetic caps, and a measurement harness that lies
- Generation metrics — groundedness, relevance, correct refusal — and two kinds of judge
- The worst result in the repo (confident made-up answers) and three levers that fix it, measured

| | |
|---|---|
| ⏱️ Time | 75–90 min |
| 🖥️ Runtime | CPU works; **T4 GPU recommended** for the generation sections (Runtime → Change runtime type) |
| 🤖 Needs the generator (Ollama) | yes — for some cells |

**Before this:** [02-chunking](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/02-chunking/colab.ipynb)

> **How to use this notebook.** Run the cells top to bottom (`Shift+Enter`). Every
> time you see **🔮 Predict**, stop and write down what you expect *before* running
> the next cell — the gap between prediction and result is where the learning is.
> Answers to the check-your-understanding questions are hidden behind ▶ toggles.

## 0. Setup

This cell clones the course repo into the Colab runtime, installs the
dependencies, and (if `USE_LLM` is ticked) installs and starts
[Ollama](https://ollama.com) with the small `qwen2.5:1.5b` generator. It is safe
to re-run. Everything in the repo's `ragkit/` library becomes importable.

If an import fails after the install, use **Runtime → Restart session** and run
this cell again.

In [ ]:
# @title ⚙️ Setup — run me first
USE_LLM = True  # @param {type:"boolean"}
BRANCH = "main"  # @param {type:"string"}

import subprocess
import sys
from pathlib import Path


def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "ragkit").is_dir() and (p / "colab" / "colab_setup.py").exists():
            return p
    return None


REPO = _find_repo()
if REPO is None:
    REPO = Path("/content/rag-from-scratch-to-production")
    if not REPO.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                BRANCH,
                "https://github.com/safar-byte/rag-from-scratch-to-production.git",
                str(REPO),
            ],
            check=True,
        )
sys.path.insert(0, str(REPO / "colab"))
from colab_setup import compare_runs, eval_frames, results_frame, setup  # noqa: F401

LLM_READY = setup(llm=USE_LLM)

## 1. Why this comes before any optimisation

Everything in lessons 04–11 is a *claim*: hybrid search helps, reranking helps,
contextual retrieval helps. Every one of those is true on *some* corpus for *some*
questions and false elsewhere. Without measurement you pick techniques by reputation —
and reputation is how people end up with a slow, expensive pipeline that is no better
than the naive one they started with.

### The two halves

Retrieval and generation fail independently, so they are measured separately. That
separation is the whole diagnostic value of the harness:

| Retrieval | Groundedness | Diagnosis |
|---|---|---|
| high | high | working |
| high | low | prompting or model problem — retrieval found it, generation fumbled it |
| low | high | generation is doing well with bad material; **fix retrieval** |
| low | low | start with retrieval; generation can't be assessed without material |

One end-to-end score collapses all four rows into "bad" and tells you nothing about
where to work.

## 2. The golden set

`ragkit/eval/golden.yaml` holds 44 questions. Each names the **document(s)** that answer
it, a few strings a correct answer should contain, and a **kind**.

In [ ]:
import pandas as pd

from ragkit.eval.golden import load_golden, validate_golden

golden = load_golden()
frame = pd.DataFrame(
    [
        {
            "id": q.id,
            "kind": q.kind,
            "question": q.question,
            "relevant": q.relevant,
            "answer_contains": q.answer_contains,
        }
        for q in golden
    ]
)
print(frame.kind.value_counts().to_string(), "\n")
pd.set_option("display.max_colwidth", 80)
frame.groupby("kind").head(2)

| kind | what it tests |
|---|---|
| `lookup` | a specific value stated in one place (a port, an exit code) |
| `conceptual` | needs explanation, not extraction |
| `multi_hop` | needs **two or more** documents |
| `vocab_mismatch` | phrased in a user's words, not the document's ("charged twice") |
| `unanswerable` | the corpus genuinely does not cover it — the right answer is a refusal |

### Designing a golden set without fooling yourself

The classic mistake is writing questions *from* the documents. You produce questions in
the document's own vocabulary, every retriever answers them perfectly, and you have
measured nothing. Better: write questions first from what a user would ask, then find
which documents answer them, and deliberately keep ones whose phrasing does not match.

**Why document-level, not chunk-level relevance?** Chunk ids change every time
`chunk_size` changes, so a chunk-level golden set silently invalidates itself during
the experiment you most want to run (lesson 02's sweep).

The harness also *validates* the set before running: a question naming a document that
no longer exists can never be answered, and shows up as a permanent, unexplained dent
in recall.

In [ ]:
problems = validate_golden(golden)
print("golden set problems:", problems or "none ✓")

## 3. Retrieval metrics, by hand

Take one question whose relevant documents are **A** and **B**, and a retriever that
returned this ranked list of documents:

```
rank:      1    2    3    4    5
returned:  C    A    D    B    E          relevant = {A, B}
```

| metric | definition | value here |
|---|---|---|
| **recall@k** | fraction of the relevant docs found in the top k | @1 = 0/2, @3 = 1/2, @5 = 2/2 |
| **precision@k** | fraction of the top k that is relevant | @1 = 0/1, @3 = 1/3, @5 = 2/5 |
| **reciprocal rank** | 1 / rank of the *first* relevant doc (MRR = mean over questions) | 1/2 |
| **nDCG@k** | discounted gain, normalised by the best possible ordering | see below |

**nDCG** credits a relevant document at rank *r* with $1/\log_2(r+1)$ — rank 1 is worth
1.0, rank 2 ≈ 0.63, rank 3 = 0.5, rank 4 ≈ 0.43 — and divides by the score of the ideal
ordering (both relevant docs at ranks 1 and 2):

$$\text{nDCG@5} = \frac{1/\log_2 3 + 1/\log_2 5}{1/\log_2 2 + 1/\log_2 3}
                 = \frac{0.631 + 0.431}{1 + 0.631} = 0.651$$

Let's implement all four from those definitions.

In [ ]:
import math


def my_recall(retrieved, relevant, k):
    return len(set(retrieved[:k]) & set(relevant)) / len(relevant)


def my_precision(retrieved, relevant, k):
    return len(set(retrieved[:k]) & set(relevant)) / len(retrieved[:k])


def my_rr(retrieved, relevant):
    return next((1 / r for r, d in enumerate(retrieved, 1) if d in relevant), 0.0)


def my_ndcg(retrieved, relevant, k):
    dcg = sum(1 / math.log2(r + 1) for r, d in enumerate(retrieved[:k], 1) if d in relevant)
    ideal = sum(1 / math.log2(r + 1) for r in range(1, min(len(relevant), k) + 1))
    return dcg / ideal


retrieved, relevant = ["C", "A", "D", "B", "E"], ["A", "B"]
for k in (1, 3, 5):
    print(
        f"k={k}: recall={my_recall(retrieved, relevant, k):.3f}  "
        f"precision={my_precision(retrieved, relevant, k):.3f}  "
        f"nDCG={my_ndcg(retrieved, relevant, k):.3f}"
    )
print("reciprocal rank =", my_rr(retrieved, relevant))

Now check against the library's implementation (`ragkit/eval/metrics.py`):

In [ ]:
from ragkit.eval.metrics import ndcg_at_k, precision_at_k, recall_at_k, reciprocal_rank

for k in (1, 3, 5):
    assert math.isclose(my_recall(retrieved, relevant, k), recall_at_k(retrieved, relevant, k))
    assert math.isclose(
        my_precision(retrieved, relevant, k), precision_at_k(retrieved, relevant, k)
    )
    assert math.isclose(my_ndcg(retrieved, relevant, k), ndcg_at_k(retrieved, relevant, k))
assert my_rr(retrieved, relevant) == reciprocal_rank(retrieved, relevant)
print("✓ hand-written metrics match ragkit")

**Why the library version is subtly different:** edge cases. What is recall when there
are *no* relevant documents (an unanswerable question)? `ragkit` returns 1.0 — "nothing
to find" — and precision returns 0.0, because any result is a false positive. Your
hand-written versions would crash with a division by zero. Getting a metric subtly
wrong does not produce an error; it produces confident nonsense for weeks, which is why
`tests/test_metrics.py` checks them against hand-computed values.

### Trap: the arithmetic cap on recall@1

A multi-hop question has **two** relevant documents. At k=1 you can return only one, so
recall@1 **cannot exceed 0.5**. Reading "multi_hop recall@1 = 0.500" as "fails half the
time" is wrong — it may be a *perfect* score. The harness prints `max_recall@1` next to
each kind for exactly this reason.

In [ ]:
print(
    "perfect retrieval for a 2-document question, recall@1 =",
    recall_at_k(["A", "B"], ["A", "B"], 1),
)

### From chunks to documents

The retriever returns *chunks*, and several chunks of one document often fill the top of
the list. The harness collapses the chunk ranking into a document ranking, keeping each
document at its best position, before computing any metric:

In [ ]:
from ragkit.eval.run import dedupe_to_documents

chunk_ranking = ["03-hybrid.md", "03-hybrid.md", "04-rerank.md", "03-hybrid.md", "01-emb.md"]
print(dedupe_to_documents(chunk_ranking))

## 4. Run the harness: the dense baseline

Build the index (if lesson 01 has not already) and run the retrieval half over all 44
questions. `retrieval_only=True` means no generation and no judge — it takes seconds.

In [ ]:
from ragkit.eval.run import run_eval
from ragkit.pipeline import RagPipeline

pipeline = RagPipeline(strategy="dense")
if pipeline.count() == 0:
    print(pipeline.ingest())

baseline = run_eval(pipeline=pipeline, retrieval_only=True, quiet=True)
overall, by_kind = eval_frames(baseline)
print(
    "ceiling reached:",
    baseline["ceiling_reached"],
    "| mean latency:",
    baseline["mean_latency_ms"],
    "ms\n",
)
display(overall.round(3))
display(by_kind.round(3))

### How to read this

- **Look for saturation.** Any kind scoring 1.000 at every k has *no headroom*: no
  technique in lessons 04–11 can show a gain on those questions. If you ran hybrid
  search tomorrow and reported "no improvement", you would be reporting a property of
  the corpus, not of hybrid search.
- **Compare `recall@1` with `max_recall@1`** before calling a multi-hop number bad.
- **`unanswerable` scores 1.000 by definition** — there was nothing to find. Ignore it
  for retrieval; it matters for generation (section 7).

The harness's `ceiling_reached` flag fires when every answerable recall@k is ≥ 0.95,
and `benchmarks/results.md` stamps such rows `⚠️ ceiling`. A working eval is supposed to
tell you uncomfortable things, and the first uncomfortable thing this one says is that
the corpus is small.

In [ ]:
import matplotlib.pyplot as plt

ax = by_kind[["recall@1", "recall@3", "recall@5"]].plot.bar(figsize=(10, 3.5), rot=0)
ax.set_ylim(0, 1.05)
ax.set_title("Dense retrieval — recall@k by question kind")
ax.axhline(1.0, color="grey", lw=0.8, ls="--")
plt.show()

### The diagnostic that tells you *what to do*: `first_relevant_rank`

When a question fails at k=5, the useful question is whether the answer was at rank 6
(→ retrieve deeper, rerank) or absent entirely (→ fix retrieval). An aggregate score
cannot distinguish those, and they need opposite responses. The harness retrieves to
depth 20 for exactly this diagnostic.

In [ ]:
per_question = pd.DataFrame(baseline["results"])
per_question["recall@5"] = per_question.retrieval.map(lambda r: r["recall@5"])
misses = per_question[(per_question.kind != "unanswerable") & (per_question["recall@5"] < 1)]
misses[["id", "kind", "question", "first_relevant_rank", "recall@5", "retrieved"]]

These are the only questions with headroom — everything later in the course will be
judged largely on them. Read each one and ask: is the right document absent, or just
ranked too low?

### Reproducibility

Two runs of the same configuration must give the same numbers, or you cannot attribute
a difference to your change rather than to noise. Temperature 0 everywhere; retrieval
is deterministic.

In [ ]:
again = run_eval(pipeline=pipeline, retrieval_only=True, quiet=True)
print("identical retrieval metrics across two runs:", again["retrieval"] == baseline["retrieval"])

## 5. Generation metrics

| metric | question it answers |
|---|---|
| **groundedness** | is every claim supported by the retrieved context? Low = hallucination |
| **relevance** | does the answer address the question? (an answer can be perfectly grounded and unresponsive) |
| **correct refusal** | on *unanswerable* questions only: did it decline, or make something up? |

Correct refusal is the most under-tested behaviour in RAG, and the fastest way to tell
a demo from a deployable system.

### Two judges, on purpose

- `DeterministicJudge` — lexical overlap and keyword checks. Crude, free, perfectly
  reproducible, needs no model.
- `LlmJudge` — a model grading against a rubric at temperature 0.

An LLM judge is a model with opinions. The only way to know whether to trust it is to
compare it against something that *cannot* have opinions. Two limits to remember: judges
favour fluent, longer answers, and **scores from different judges are not comparable**
— changing the judge model invalidates every historical row.

Let's grade some hand-written answers with the deterministic judge first.

In [ ]:
from dataclasses import asdict

from ragkit.eval.judge import DeterministicJudge, looks_like_refusal

judge = DeterministicJudge()
q_port = next(q for q in golden if q.id == "q01")
q_fake = next(q for q in golden if "SHARD_REPLICATION_FACTOR" in q.question)
contexts = pipeline.retrieve(q_port.question, top_k=3)

samples = [
    (q_port, "The query service listens on port 8788 by default [1]."),
    (q_port, "It probably uses port 8080 like most web services."),
    (q_fake, "The default value of SHARD_REPLICATION_FACTOR is 1."),
    (q_fake, "The provided context does not contain this information."),
]
pd.DataFrame(
    [
        {"question": q.question[:45], "answer": a, **asdict(judge.grade(q, a, contexts))}
        for q, a in samples
    ]
).drop(columns=["notes"])

### The bug that made a correct refusal look like a lie

Refusal is detected by substring markers (`looks_like_refusal`). An early version
matched `"not provided"` but **not** `"does not provide"` — so the model's genuine
refusal *"The passage does not provide any information about the population of Tokyo"*
was scored as a confabulation. The reported refusal rate was **0.000**; the true value
was **0.300**.

🔮 **Predict:** which of these does the current detector catch?

In [ ]:
for text in [
    "The passage does not provide any information about the population of Tokyo.",
    "That information is not provided in the context.",
    "I'm not sure, but it is probably around 14 million.",
    "The context doesn't mention a cloud provider.",
    "Sorry, I can't help with that.",
]:
    print(f"{looks_like_refusal(text)!s:5}  {text}")

Note the last one: a refusal phrased in a way nobody listed. **A gap in the detector is
indistinguishable from a failure in the model** once it reaches the results table. The
measurement harness is also code, and it can lie with a perfectly plausible number.
`tests/test_refusal_detection.py` pins the model's real refusal phrasings so the fix
cannot be quietly undone.

## 6. Generate and grade

Now run real generation. We use a focused subset so it runs in a few minutes even on
CPU: **all 10 unanswerable questions** (does it refuse?) plus **5 answerable ones**
(does it *wrongly* refuse?). That second group is the control: a refusal metric without
a false-refusal check would make "refuse everything" look perfect.

In [ ]:
import time

from ragkit.generate import abstain, generate_answer

unanswerable = [q for q in golden if q.kind == "unanswerable"]
answerable = [q for q in golden if q.kind == "lookup"][:5]
subset = unanswerable + answerable


def run_config(name, *, strict=False, floor=None):
    rows, started = [], time.perf_counter()
    for q in subset:
        ctx = pipeline.retrieve(q.question, top_k=5)
        decision = abstain.decide(ctx, floor=floor) if floor is not None else None
        if decision is not None and decision.abstain:
            text = abstain.REFUSAL_TEXT  # refused without a model call
        else:
            text = generate_answer(q.question, ctx, pipeline.generator, strict=strict).text
        rows.append(
            {
                "config": name,
                "id": q.id,
                "kind": q.kind,
                "question": q.question,
                "refused": looks_like_refusal(text),
                "answer": text,
            }
        )
    print(f"{name}: {time.perf_counter() - started:.0f}s")
    return rows


if LLM_READY:
    default_rows = run_config("default prompt")
    display(pd.DataFrame(default_rows)[["kind", "question", "refused", "answer"]])

Read the unanswerable answers carefully. When this course was built, the best pipeline
— top of every retrieval metric — said things like:

| Question (nothing in the corpus answers it) | What it said |
|---|---|
| Default of `SHARD_REPLICATION_FACTOR`? | **"The default value … is 1."** |
| Default of `MAX_CHUNK_BYTES`? | **"is 1024 bytes."** |
| Which cloud provider? | **"AWS (Amazon Web Services)"** |

`SHARD_REPLICATION_FACTOR` does not exist anywhere. It was invented for the golden set
precisely because it *sounds* like it should. The system produced a plausible default
and stated it as confidently as the answers it got right.

Note what would have happened without this metric: every retrieval number improved
across lessons 04 and 05, the table looked like steady progress, and nobody would have
asked what the system does when it does not know.

### Three levers

1. **A strict prompt** (`STRICT_SYSTEM_PROMPT`) — names the failure mode, gives a
   procedure ("do the passages actually contain the answer?"), and states which error
   is worse. Free.
2. **A score floor** — if the best dense similarity is below 0.60, refuse *without
   calling the model*. One float comparison. (Calibrated in lesson 12.)
3. **A grader** that reads the passages (CRAG, lesson 10). One model call.

Let's measure the first two.

In [ ]:
from ragkit.generate import STRICT_SYSTEM_PROMPT

print(STRICT_SYSTEM_PROMPT)

In [ ]:
if LLM_READY:
    all_rows = (
        default_rows
        + run_config("strict prompt", strict=True)
        + run_config("score floor 0.60", floor=0.60)
    )
    results = pd.DataFrame(all_rows)
    summary = results.groupby("config").apply(
        lambda g: pd.Series(
            {
                "correct refusal (unanswerable)": g[g.kind == "unanswerable"].refused.mean(),
                "false refusals (answerable)": f"{g[g.kind != 'unanswerable'].refused.sum()}/{(g.kind != 'unanswerable').sum()}",
            }
        ),
        include_groups=False,
    )
    display(summary)

**Compare with what the repo measured** (same model, same corpus):

| Lever | Correct refusal | False refusals on answerable | Cost |
|---|---|---|---|
| default prompt | 0.300 | 0/5 | — |
| **strict prompt** | **0.900** | **1/5 (20%)** | free |
| score floor 0.60 | 0.500 | 0/34 | one float comparison |

Your numbers may differ slightly (different hardware can change a small model's output
even at temperature 0). The shape is what matters:

- **Prompting is the largest single lever, and it is free** — an earlier draft of this
  repo claimed the opposite, before measuring.
- In the repo's run it paid by **over-refusing**: one valid question in five declined,
  including "What port does the query service listen on by default?", which the
  corpus plainly answers. Check your own run below — and remember that 5 control
  questions is a small sample (the repo's own refusal number was once wrong because
  it was measured over only 3 questions).
- The score floor refuses nothing valid. Its catch rate depends on which retriever
  produced the scores (the repo's 0.500 was over the reranked pipeline; this notebook
  uses dense), which is one more reason lesson 12 calibrates it properly.

Which trade-off is right depends on whether a wrong answer or a missing answer costs
more in your application. **The only reason that choice is visible is that both
directions were measured.**

Inspect the strict prompt's false refusal(s):

In [ ]:
if LLM_READY:
    wrong = results[
        (results.config == "strict prompt") & (results.kind != "unanswerable") & results.refused
    ]
    if wrong.empty:
        print(
            "No false refusals in this run — the 1-in-5 from the repo's run did not "
            "reproduce here. With 5 control questions, one answer flipping is the "
            "whole difference."
        )
    else:
        display(wrong[["question", "answer"]])

## 7. The LLM judge

The deterministic judge cannot tell a well-grounded paraphrase from a lucky word match.
The LLM judge can — if you can trust it. Let's grade the same answers with both and look
at where they disagree.

In [ ]:
from ragkit.eval.judge import JUDGE_SYSTEM, LlmJudge

print(JUDGE_SYSTEM)

In [ ]:
if LLM_READY:
    llm_judge = LlmJudge(pipeline.generator)
    rows = []
    for q, a in samples:
        ctx = pipeline.retrieve(q.question, top_k=3)
        det, llm = judge.grade(q, a, ctx), llm_judge.grade(q, a, ctx)
        rows.append(
            {
                "answer": a[:60],
                "det. grounded": det.groundedness,
                "LLM grounded": llm.groundedness,
                "LLM relevance": llm.relevance,
                "LLM notes": llm.notes,
            }
        )
    display(pd.DataFrame(rows))

A 1.5B-parameter model is a weak judge. When this notebook was written it scored the
*correct refusal* (last row) **0** for groundedness — even though its own rubric says
"an answer that correctly states the context does not cover the question scores 1".
One row, confidently wrong, in a perfectly plausible table. That is the lesson: **hand-label ~20 answers, score them with both
judges, and check agreement before trusting any judge's numbers.** If they disagree
wildly, the judge is miscalibrated and its numbers are not facts. On the cloud profile
the judge is Claude (cheap model), which is far more reliable — but still worth
validating.

## 8. The full harness, and recording a result

Everything above is what `python -m ragkit.eval.run` does in one command. The flags map
directly onto what you just did by hand:

```bash
python -m ragkit.eval.run --retrieval-only                       # section 4
python -m ragkit.eval.run --strict-prompt --no-llm-judge         # section 6
python -m ragkit.eval.run --score-floor 0.60                     # section 6
python -m ragkit.eval.run --write --label "my change"            # append to results.md
```

Each later lesson adds a row to `benchmarks/results.md` — *including when it got worse*.
Here is the row this run would append (we do not write it: changes in Colab vanish with
the runtime unless you commit and push them):

In [ ]:
from ragkit.eval.report import build_row

print(build_row(baseline, "dense baseline (Colab)"))

In [ ]:
!{sys.executable} -m ragkit.eval.run --retrieval-only --strategy dense 2>&1 | tail -25

Take a look at the results table the whole course builds — it reads as one story of
what each technique was actually worth:

In [ ]:
from IPython.display import Markdown

display(Markdown(Path("benchmarks/results.md").read_text()))

## 9. Check your understanding

**1. Retrieval recall is high but groundedness is low. Where do you work?**

<details><summary>▶ Show answer</summary>

On generation — the prompt or the model. Retrieval found the material; the model
fumbled it (ignored it, embellished it, or answered from memory).

</details>

**2. A multi-hop row shows recall@1 = 0.500. Is retrieval broken?**

<details><summary>▶ Show answer</summary>

Not necessarily. With two relevant documents, recall@1 is capped at 0.5 by
arithmetic. Compare it with `max_recall@1`; if they are equal, that is a perfect
score.

</details>

**3. Why does the refusal experiment include answerable questions?**

<details><summary>▶ Show answer</summary>

As a control. A system that refuses everything scores a perfect correct-refusal
rate. Measuring false refusals on answerable questions exposes that trade-off — it
is how we know the strict prompt costs one valid answer in five.

</details>

**4. You switch the LLM judge from one model to another and groundedness rises from 0.71 to 0.83. Did the system get better?**

<details><summary>▶ Show answer</summary>

You cannot tell. Scores from different judges are not comparable. Re-score the old
configuration with the new judge, or compare both judges against hand labels.

</details>

## ✍️ Exercises

1. **Break a metric and watch the tests catch it.** In a copy of `my_ndcg`, forget to
   divide by `ideal`. What numbers come out? Then run
   `!{sys.executable} -m pytest tests/test_metrics.py -q` — the real tests use
   hand-computed values for exactly this.
2. **Write a question that breaks it.** Add a question to a copy of the golden set that
   you predict dense retrieval will miss (hint: vocabulary mismatch, or a fact split
   across two documents). Run it through `pipeline.retrieve` and check
   `first_relevant_rank`.
3. **Validate the judge.** Hand-label the 15 answers from section 6 as grounded / not
   grounded, then compare with `LlmJudge`. What is the agreement rate?
4. **Grow the corpus** (the repo's #1 open problem). Everything is bounded by its size;
   how would you add documents and questions without writing questions *from* the
   documents?

## Key takeaways

1. Measure retrieval and generation separately — that is what tells you where to work.
2. recall@k is the ceiling on everything downstream; MRR rewards the top hit; nDCG keeps caring after it.
3. Watch for saturation (no headroom) and arithmetic caps (multi-hop recall@1 ≤ 0.5).
4. Measure refusals *and* false refusals — the strict prompt took refusal 0.3 → 0.9 for free, at the cost of 1 in 5 valid answers.
5. The harness is code too: a detector gap reported 0.000 when the truth was 0.300.

## Next

**[Lesson 04 — Hybrid search and RRF](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/04-hybrid-search-rrf/colab.ipynb)**